# Homework 6: Decision Trees and Ensemble Learning

**Course:** Machine Learning Zoomcamp 2026  
**Module:** Decision Trees and Ensemble Learning

## Objective

Build regression models to predict car fuel efficiency (`fuel_efficiency_mpg`) using the 2026 car fuel-efficiency dataset.

## Models and techniques

- Decision Tree Regressor
- Random Forest Regressor
- Hyperparameter tuning
- Feature importance
- XGBoost

## Dataset

`car_fuel_efficiency_2026.csv`

## Evaluation Metric

Root Mean Squared Error (RMSE)

## 2. Environment Setup

### 2.1 Objective

Before building the regression models, we need to prepare the Python environment with the libraries required for data manipulation, feature engineering, model development, hyperparameter tuning, and evaluation.

### 2.2 Libraries Used

- **Pandas:** Data loading, manipulation, and inspection.
- **NumPy:** Numerical computations.
- **Scikit-learn:** Dataset splitting, feature vectorization, regression models, and evaluation.
- **XGBoost:** Gradient boosting regression.

### 2.3 Reproducibility

A fixed random seed will be used wherever specified in the assignment to ensure consistent and reproducible results.

In [2]:
# Data manipulation and numerical operations
import pandas as pd
import numpy as np

# Data preparation and feature engineering
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

# Machine learning models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Model evaluation
from sklearn.metrics import mean_squared_error

# Reproducibility
SEED = 1

## 3. Data Understanding

### 3.1 Dataset Overview

The dataset contains information about cars and their fuel efficiency. It includes vehicle specifications and characteristics that can help explain differences in fuel consumption.

The target variable for this regression task is `fuel_efficiency_mpg`.

### 3.2 Analytical Objectives

Before training the models, we will:

- Load and inspect the dataset.
- Understand its dimensions and structure.
- Identify numerical and categorical features.
- Examine missing values.
- Understand the distribution of the target variable.

This initial exploration will help us understand the data and prepare it correctly for model development.

In [3]:
# Load the dataset
df = pd.read_csv("car_fuel_efficiency_2026.csv")

# Display the first five records
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
# Dataset dimensions
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

# Column names and data types
df.info()

Number of rows: 10000
Number of columns: 11
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   model_year           10000 non-null  int64  
 1   origin               10000 non-null  str    
 2   fuel_type            10000 non-null  str    
 3   drivetrain           10000 non-null  str    
 4   num_doors            10000 non-null  int64  
 5   engine_displacement  10000 non-null  int64  
 6   num_cylinders        10000 non-null  int64  
 7   horsepower           9123 non-null   float64
 8   vehicle_weight       10000 non-null  int64  
 9   acceleration         9736 non-null   float64
 10  fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(3), int64(5), str(3)
memory usage: 859.5 KB


## 3. Data Preparation

The dataset must be prepared before training the regression models.

The following preparation steps are specified in the homework:

1. Fill all missing values with zero.
2. Split the dataset into training, validation, and test sets using the specified random seed and proportions.
3. Separate the target variable from the input features.
4. Convert the dataframes into numerical matrices using `DictVectorizer(sparse=True)`.

In [5]:
# Fill missing values with zero as specified in the homework
df = df.fillna(0)

# Verify that no missing values remain
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [6]:
# Split into full training and test datasets
df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=1
)

# Split full training data into training and validation datasets
df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=1
)

# Check dataset dimensions
print("Training set:", df_train.shape)
print("Validation set:", df_val.shape)
print("Test set:", df_test.shape)

Training set: (6000, 11)
Validation set: (2000, 11)
Test set: (2000, 11)


In [7]:
# Separate target variable from input features
y_train = df_train["fuel_efficiency_mpg"]
y_val = df_val["fuel_efficiency_mpg"]
y_test = df_test["fuel_efficiency_mpg"]

# Remove target variable from feature datasets
X_train = df_train.drop(columns=["fuel_efficiency_mpg"])
X_val = df_val.drop(columns=["fuel_efficiency_mpg"])
X_test = df_test.drop(columns=["fuel_efficiency_mpg"])

# Verify dimensions
print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)
print("Test features:", X_test.shape)

Training features: (6000, 10)
Validation features: (2000, 10)
Test features: (2000, 10)


In [8]:
# Initialize DictVectorizer
dv = DictVectorizer(sparse=True)

# Convert dataframes into dictionaries
train_dict = X_train.to_dict(orient="records")
val_dict = X_val.to_dict(orient="records")
test_dict = X_test.to_dict(orient="records")

# Fit on training data and transform all datasets
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)
X_test = dv.transform(test_dict)

# Check matrix dimensions
print("Training matrix:", X_train.shape)
print("Validation matrix:", X_val.shape)
print("Test matrix:", X_test.shape)

Training matrix: (6000, 16)
Validation matrix: (2000, 16)
Test matrix: (2000, 16)


## 4. Decision Tree Regression

### 4.1 Objective

Train a Decision Tree Regressor to predict car fuel efficiency using the prepared training dataset.

### 4.2 Model Configuration

The homework specifies a maximum tree depth of 1.

At this depth, the model makes a single split using the feature that provides the best reduction in prediction error.

### 4.3 Evaluation

Identify the feature selected at the root node and map it back to its original dataset column if it is a one-hot encoded feature.

In [9]:
# Initialize Decision Tree Regressor
dt = DecisionTreeRegressor(max_depth=1)

# Train the model
dt.fit(X_train, y_train)

# Identify the feature used at the root split
feature_index = dt.tree_.feature[0]
split_feature = dv.get_feature_names_out()[feature_index]

# Map encoded feature to original column name
original_feature = split_feature.split("=")[0]

print("Selected feature:", split_feature)
print("Original feature:", original_feature)

Selected feature: model_year
Original feature: model_year


### 4.4 Results and Interpretation

The Decision Tree Regressor was trained with `max_depth=1`.

The model selected `model_year` as the root splitting feature.

**Answer – Q1: `model_year`**

This indicates that, among the available predictors, model year provided the best reduction in prediction error for the first split.

## 5. Random Forest Regression

### 5.1 Objective

A Random Forest combines multiple decision trees to improve predictive performance and reduce the instability of an individual tree.

In this task, we train a Random Forest Regressor to predict car fuel efficiency and evaluate its performance using Root Mean Squared Error (RMSE) on the validation dataset.

### 5.2 Model Configuration

The homework specifies:

- Number of estimators: 10
- Random state: 1
- Number of jobs: -1

The trained model will be evaluated on the validation dataset.

In [10]:
# Initialize Random Forest Regressor
rf = RandomForestRegressor(
    n_estimators=10,
    random_state=SEED,
    n_jobs=-1
)

# Train the model
rf.fit(X_train, y_train)

# Generate predictions on validation data
y_pred = rf.predict(X_val)

# Calculate validation RMSE
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 1.8332


### 5.3 Results and Interpretation

The Random Forest Regressor was trained using 10 decision trees with a fixed random state of 1.

The model achieved a validation RMSE of **1.8332**.

**Answer – Q2: 1.837** (closest available option).

The result provides a baseline for evaluating the impact of increasing the number of trees in the next experiment.

## 6. Hyperparameter Tuning

### 6.1 Number of Estimators (Q3)

The `n_estimators` parameter controls the number of decision trees in a Random Forest.

Increasing the number of trees can improve prediction stability, but it also increases computational cost.

In this experiment, we compare four configurations using the validation dataset:

- 10 estimators
- 50 estimators
- 100 estimators
- 150 estimators

The random state is fixed at 1. Validation RMSE is used to identify the best configuration.

In [11]:
# Compare different numbers of estimators
results = []

for n in [10, 50, 100, 150]:
    model = RandomForestRegressor(
        n_estimators=n,
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    score = np.sqrt(mean_squared_error(y_val, predictions))

    results.append({
        "n_estimators": n,
        "RMSE": score
    })

# Display comparison
results_df = pd.DataFrame(results)
results_df

,n_estimators,RMSE
0,10,1.833220
1,50,1.773988
2,100,1.769786
3,150,1.769198


### 6.2 Results and Interpretation

The Random Forest model was evaluated using four different values of `n_estimators`.

The validation RMSE decreased as the number of estimators increased from 10 to 150.

The lowest RMSE was **1.769198**, achieved with 150 estimators.

**Answer – Q3: 150**

Among the tested configurations, 150 estimators provided the best validation performance.

### 6.3 Maximum Depth Tuning (Q4)

The `max_depth` parameter controls how deep each decision tree can grow.

- A shallow tree may underfit by failing to capture important patterns.
- A very deep tree may overfit the training data.
- Selecting an appropriate depth helps balance model complexity and predictive performance.

We will evaluate maximum depths of 10, 15, 20, and 25, using 10, 50, 100, and 150 estimators for each depth.

The mean validation RMSE across the four estimator configurations will determine the best maximum depth.

In [12]:
# Compare maximum depths using different estimator counts
depth_results = []

for depth in [10, 15, 20, 25]:
    rmse_scores = []

    for n in [10, 50, 100, 150]:
        model = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=SEED,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, predictions))
        rmse_scores.append(rmse)

    depth_results.append({
        "max_depth": depth,
        "mean_RMSE": np.mean(rmse_scores)
    })

# Display mean RMSE for each depth
depth_results_df = pd.DataFrame(depth_results)
depth_results_df

,max_depth,mean_RMSE
0,10,1.754392
1,15,1.784224
2,20,1.786771
3,25,1.786305


### 6.4 Results and Interpretation

The Random Forest model was evaluated across four maximum depth values, with four different estimator configurations for each depth.

The mean validation RMSE was calculated for each maximum depth.

The lowest mean RMSE was **1.754392**, achieved at a maximum depth of 10.

**Answer – Q4: 10**

Among the tested configurations, a maximum depth of 10 provided the best average validation performance.

## 7. Feature Importance

### 7.1 Objective

Tree-based models provide feature importance scores that indicate how much each feature contributes to reducing prediction error across the trees.

In this experiment, we will use the Random Forest Regressor's `feature_importances_` attribute to identify the most important predictor of car fuel efficiency.

### 7.2 Model Configuration

The homework specifies:

- Number of estimators: 10
- Maximum depth: 20
- Random state: 1

We will extract the feature importance scores, map encoded features to their original names, and identify the most important feature.

In [13]:
# Initialize Random Forest with the specified parameters
rf_importance = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=SEED,
    n_jobs=-1
)

# Train the model
rf_importance.fit(X_train, y_train)

# Extract feature importance
feature_importance_df = pd.DataFrame({
    "feature": dv.get_feature_names_out(),
    "importance": rf_importance.feature_importances_
})

# Sort features by importance
feature_importance_df = feature_importance_df.sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)

feature_importance_df

,feature,importance
0,model_year,0.296107
1,vehicle_weight,0.203313
2,fuel_type=Gasoline,0.160712
3,horsepower,0.078271
4,engine_displacement,0.061281
5,acceleration,0.054998
6,origin=USA,0.042152
7,drivetrain=Front-wheel drive,0.020062
8,num_doors,0.019401
9,fuel_type=Diesel,0.016571


### 7.3 Results and Interpretation

The Random Forest Regressor was trained using 10 estimators, a maximum depth of 20, and random state 1.

Feature importance scores were extracted using the `feature_importances_` attribute.

Among the four features specified in the homework, `vehicle_weight` achieved the highest importance score of **0.203313**.

**Answer – Q5: `vehicle_weight`**

This indicates that vehicle weight contributes more to reducing prediction error than horsepower, engine displacement, and acceleration among the four specified features.

## 8. XGBoost Regression

### 8.1 Objective

XGBoost is a gradient boosting algorithm that builds an ensemble of decision trees sequentially. Each new tree attempts to improve the predictions made by the previous trees.

In this experiment, we compare two learning rates (`eta`) to understand their impact on validation RMSE.

### 8.2 Model Configuration

The homework specifies the following parameters:

- Maximum depth: 6
- Minimum child weight: 1
- Objective: `reg:squarederror`
- Number of boosting rounds: 100
- Seed: 1
- Number of threads: 8

We will compare:

- `eta = 0.3`
- `eta = 0.1`

The model with the lower validation RMSE will be considered better.

In [14]:
# Import XGBoost
import xgboost as xgb

# Create DMatrix for training and validation data
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# Create watchlist
watchlist = [
    (dtrain, "train"),
    (dval, "validation")
]

In [15]:
# XGBoost parameters with eta = 0.3
xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": SEED,
    "verbosity": 1
}

# Train the model for 100 boosting rounds
model_eta_03 = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    verbose_eval=False
)

# Calculate validation RMSE
predictions_eta_03 = model_eta_03.predict(dval)

rmse_eta_03 = np.sqrt(
    mean_squared_error(y_val, predictions_eta_03)
)

print(f"Validation RMSE (eta=0.3): {rmse_eta_03:.6f}")

Validation RMSE (eta=0.3): 1.826458


In [16]:
# XGBoost parameters with eta = 0.1
xgb_params_eta_01 = {
    "eta": 0.1,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": SEED,
    "verbosity": 1
}

# Train the model for 100 boosting rounds
model_eta_01 = xgb.train(
    xgb_params_eta_01,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    verbose_eval=False
)

# Calculate validation RMSE
predictions_eta_01 = model_eta_01.predict(dval)

rmse_eta_01 = np.sqrt(
    mean_squared_error(y_val, predictions_eta_01)
)

print(f"Validation RMSE (eta=0.1): {rmse_eta_01:.6f}")

Validation RMSE (eta=0.1): 1.724810


### 8.3 Results and Interpretation

Two XGBoost models were trained using 100 boosting rounds, with identical configurations except for the learning rate (`eta`).

| Learning Rate | Validation RMSE |
|---|---:|
| 0.3 | 1.826458 |
| 0.1 | 1.724810 |

The model with `eta=0.1` achieved the lower validation RMSE.

**Answer – Q6: 0.1**

The lower learning rate provided better validation performance in this experiment.

## 9. Conclusion

### 9.1 Homework Results

This project explored decision trees, random forests, hyperparameter tuning, feature importance, and XGBoost for predicting car fuel efficiency.

The experiments produced the following results:

| Question | Experiment | Result |
|---|---|---|
| Q1 | Decision Tree splitting feature | `model_year` |
| Q2 | Random Forest validation RMSE | 1.8332 (closest option: 1.837) |
| Q3 | Best number of estimators | 150 |
| Q4 | Best maximum depth | 10 |
| Q5 | Most important feature among the given options | `vehicle_weight` |
| Q6 | Best XGBoost learning rate | 0.1 |

### 9.2 Key Observations

- The Decision Tree selected `model_year` for its root split.
- Increasing the number of estimators improved validation performance across the tested configurations.
- A maximum depth of 10 achieved the lowest mean RMSE in the depth-tuning experiment.
- Vehicle weight was the most important feature among the four specified options.
- XGBoost with a learning rate of 0.1 outperformed the configuration with a learning rate of 0.3.

### 9.3 Final Takeaway

The experiments demonstrate how tree-based ensemble methods can capture relationships between vehicle characteristics and fuel efficiency.

Hyperparameter tuning and feature importance analysis provided additional insight into model performance and the contribution of individual predictors.

All experiments followed the dataset preparation and evaluation requirements specified in ML Zoomcamp 2026 Homework 6.

-- By Shubham kulkarni